# Uncertainty and calibration

- **Question:** Do the empirical P10/P50/P90 ranges achieve useful out-of-time coverage without becoming uninformatively wide?
- **Data used:** Persisted Phase 4 candidate predictions and live projection-board status from the canonical DuckDB warehouse.
- **Unit of observation:** One player-season prediction for one model family, position, and target.
- **Target:** Next-season fantasy points per active game, games active, or total fantasy points.
- **Feature cutoff:** Point models and residual calibration use only seasons earlier than the inference season.
- **Validation strategy:** Evaluate ordered intervals on 2020-2024 validation and the untouched 2025 test, including position, experience, and projection-tier segments.
- **Interpretation caveat:** These are empirical residual intervals, not guarantees. Rookie heuristic ranges are explicitly unvalidated and uncalibrated.

## Signed residual calibration

For an earlier out-of-fold prediction, the signed residual is `actual - prediction`. Phase 4 adds earlier residual quantiles to the current point estimate. This preserves asymmetric misses and allows P50 to correct persistent bias. Coverage must be read together with interval width and pinball loss.

In [ ]:
from __future__ import annotations

from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from fantasy_draft_ai.models.player_projection.evaluation import (
    assign_projection_tiers,
    interval_metrics,
)


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    raise FileNotFoundError("Could not find the repository root.")


def table_exists(connection: duckdb.DuckDBPyConnection, table_name: str) -> bool:
    row = connection.execute(
        "SELECT count(*) FROM information_schema.tables WHERE table_name = ?",
        [table_name],
    ).fetchone()
    return bool(row and row[0])


PROJECT_ROOT = find_project_root()
WAREHOUSE_PATH = PROJECT_ROOT / "data" / "warehouse" / "fantasy_football.duckdb"
print({"warehouse_ready": WAREHOUSE_PATH.is_file()})

In [ ]:
interval_predictions = pd.DataFrame()
live_status = pd.DataFrame()
if WAREHOUSE_PATH.is_file():
    with duckdb.connect(str(WAREHOUSE_PATH), read_only=True) as connection:
        if table_exists(connection, "player_projection_predictions"):
            interval_predictions = connection.execute(
                """
                SELECT player_id, prediction_season, position, target_name, model_family,
                       fold_label, predicted_value, p10, p50, p90, actual_value,
                       experience_group, training_max_season
                FROM player_projection_predictions
                WHERE actual_value IS NOT NULL
                ORDER BY prediction_season, position, target_name, model_family, player_id
                """
            ).df()
        if table_exists(connection, "player_projection_board"):
            live_status = connection.execute(
                """
                SELECT prediction_season, position, prediction_status, count(*) AS players
                FROM player_projection_board
                GROUP BY prediction_season, position, prediction_status
                ORDER BY prediction_season, position, prediction_status
                """
            ).df()

if interval_predictions.empty:
    print("No evaluable Phase 4 intervals are stored yet. Run player-model training first.")
else:
    display(interval_predictions.head())

## Check invariants before metrics

Every interval must be finite and ordered, and every stored training cutoff must precede the season it predicts. Fix a failed invariant at its source; do not sort endpoints afterward to make a chart look valid.

In [ ]:
if not interval_predictions.empty:
    required_interval = interval_predictions[["p10", "p50", "p90"]].notna().all(axis=1)
    evaluable_intervals = interval_predictions.loc[required_interval].copy()
    assert (evaluable_intervals["p10"] <= evaluable_intervals["p50"]).all()
    assert (evaluable_intervals["p50"] <= evaluable_intervals["p90"]).all()
    assert (
        evaluable_intervals["training_max_season"] < evaluable_intervals["prediction_season"]
    ).all()
    print({"rows_with_intervals": len(evaluable_intervals), "ordered": True, "cutoff_safe": True})
else:
    evaluable_intervals = pd.DataFrame()

In [ ]:
metric_rows: list[dict[str, object]] = []
if not evaluable_intervals.empty:
    for keys, group in evaluable_intervals.groupby(
        ["fold_label", "position", "target_name", "model_family"], dropna=False
    ):
        metrics = interval_metrics(group["actual_value"], group["p10"], group["p50"], group["p90"])
        metric_rows.append(
            {
                "fold_label": keys[0],
                "position": keys[1],
                "target_name": keys[2],
                "model_family": keys[3],
                **metrics,
            }
        )
interval_summary = pd.DataFrame(metric_rows)
if interval_summary.empty:
    print("No complete interval rows are available for calibration metrics.")
else:
    display(interval_summary)

## Segment the calibration

An acceptable overall number can hide a weak position, experience group, or projection tier. Tiers below are assigned within season, position, target, and model family so a QB total is never ranked directly against a TE points-per-game estimate.

In [ ]:
segment_summary = pd.DataFrame()
if not evaluable_intervals.empty:
    segmented = evaluable_intervals.copy()
    segmented["projection_tier"] = "unassigned"
    tier_keys = ["prediction_season", "position", "target_name", "model_family"]
    for _, indexes in segmented.groupby(tier_keys, dropna=False).groups.items():
        labels = assign_projection_tiers(
            segmented.loc[indexes, "predicted_value"],
            entity_ids=segmented.loc[indexes, "player_id"],
        )
        segmented.loc[indexes, "projection_tier"] = labels

    rows: list[dict[str, object]] = []
    for keys, group in segmented.groupby(
        [
            "fold_label",
            "position",
            "target_name",
            "model_family",
            "experience_group",
            "projection_tier",
        ],
        dropna=False,
    ):
        rows.append(
            {
                "fold_label": keys[0],
                "position": keys[1],
                "target_name": keys[2],
                "model_family": keys[3],
                "experience_group": keys[4],
                "projection_tier": keys[5],
                **interval_metrics(group["actual_value"], group["p10"], group["p50"], group["p90"]),
            }
        )
    segment_summary = pd.DataFrame(rows)

if segment_summary.empty:
    print("No segmented calibration results are available.")
else:
    display(segment_summary)

In [ ]:
if not interval_summary.empty:
    figure, axis = plt.subplots(figsize=(8, 5))
    for position, group in interval_summary.groupby("position"):
        axis.scatter(
            group["mean_interval_width_p10_p90"],
            group["empirical_coverage_p10_p90"],
            label=position,
        )
    axis.axhline(0.80, color="black", linestyle="--", linewidth=1, label="nominal 80%")
    axis.set(
        title="Empirical coverage must be read with interval width",
        xlabel="Mean P10-P90 width",
        ylabel="Observed P10-P90 coverage",
    )
    axis.legend()
    plt.tight_layout()
    plt.show()

## Live-status boundary

The board must distinguish learned veteran rows from transparent fallback rows. In particular, a rookie fallback cannot inherit a veteran calibration claim merely because the UI has three interval columns.

In [ ]:
if live_status.empty:
    print("No live Phase 4 board is stored yet.")
else:
    display(live_status)

## Exercise

Choose one position and target. Compare validation coverage, test coverage, and mean width; then repeat by experience group or projection tier. Explain why a wider interval can improve coverage without improving usefulness. Finally, verify that no 2025 residual could have been used to calibrate a 2025 prediction and locate the unvalidated rookie-fallback label.